# Stage 1: Sliding-Window Transcript Chunking

**Author:** Rhesa Muhammad Ramadhan (29024002@mahasiswa.itb.ac.id)  
**License:** GNU AGPL

**Purpose:** Segment raw transcript `.txt` files into fixed-size word windows so long interviews can be processed locally in manageable units.

**Input:** Plain-text transcripts from `inp_folder`, plus chunking parameters such as `WINDOW_SIZE`, `OVERLAP_PERCENTAGE`, `MODE`, and optional padding settings.

**Process:** Clean timestamp-like artifacts, compute stride from the selected overlap rule, split each transcript into sequential word-based windows, assign a `window_address`, and export the chunks to CSV.

**Output:** One CSV per transcript with `window_address` and `chunked_text` columns, saved to `out_folder`.

In [ ]:
import os
import csv
import math
import re

# Define modular parameters for chunking
WINDOW_SIZE = 400  # Number of words in each chunk
OVERLAP_PERCENTAGE = 0.00  # Overlap percentage
MODE = 2  # Mode: 1 for fixed integer stride, 2 for overlap-based stride
ENABLE_PADDING = False  # Toggle to enable or disable padding
PADDING_MARKER = "N/A"  # Use "N/A" as the padding marker

inp_folder = "./stage2_500-word_prompt/qwen2.5-7b"
out_folder = "./stage2_500-word_prompt/qwen2.5-7b/qualitative_w400_ov0.00"

# Calculate STRIDE based on mode
if MODE == 1:
    STRIDE = 25
elif MODE == 2:
    STRIDE = math.ceil(WINDOW_SIZE * (1 - OVERLAP_PERCENTAGE))
else:
    raise ValueError("Invalid MODE. Use 1 for fixed stride or 2 for overlap-based stride.")

def calculate_dynamic_padding(text_length, window_size, stride):
    num_chunks = math.ceil(text_length / stride)
    total_length_needed = (num_chunks - 1) * stride + window_size
    total_padding_needed = total_length_needed - text_length
    padding_per_side = math.ceil(total_padding_needed / 2)
    min_padding = window_size - stride
    return max(padding_per_side, min_padding)

def clean_transcript_text(text):
    """
    Clean transcript by removing standalone time brackets (e.g., 06:07),
    while preserving actual content.
    """
    # Remove time patterns like 00:12, 6:30, etc., only if on their own or with speaker label
    cleaned = re.sub(r'(?<=\b)(\d{1,2}:\d{2})(?=\b)', '', text)
    return cleaned.strip()

def chunk_transcript(input_text, window_size=WINDOW_SIZE, stride=STRIDE, enable_padding=ENABLE_PADDING):
    # Clean and sanitize text
    input_text = clean_transcript_text(input_text.strip())
    words = input_text.split()

    # Handle short text: always return one chunk if shorter than window
    if len(words) < window_size:
        return [" ".join(words)]

    if enable_padding:
        padding_size = calculate_dynamic_padding(len(words), window_size, stride)
        words = [PADDING_MARKER] * padding_size + words + [PADDING_MARKER] * padding_size

    chunks = []
    for i in range(0, len(words) - window_size + 1, stride):
        chunk = words[i:i + window_size]
        chunk_text = " ".join(chunk)
        chunks.append(chunk_text)

    return chunks

def save_chunks_to_csv(chunks, output_file):
    with open(output_file, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.writer(csvfile, delimiter=';')
        writer.writerow(['window_address', 'chunked_text'])
        for index, chunk in enumerate(chunks, start=1):
            writer.writerow([index, chunk])

def batch_chunk_transcripts(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            print(f"Processing transcript '{filename}'...")

            input_path = os.path.join(input_folder, filename)
            output_filename = f"chunked_{os.path.splitext(filename)[0]}.csv"
            output_path = os.path.join(output_folder, output_filename)

            with open(input_path, 'r', encoding='utf-8') as file:
                # raw_transcript = file.read()
                raw_transcript = file.read().strip()

            chunks = chunk_transcript(raw_transcript)
            save_chunks_to_csv(chunks, output_path)

            print(f"DONE! Saved as '{output_filename}'")

if __name__ == "__main__":
    input_folder = inp_folder
    output_folder = out_folder

    print("Starting Phase 2 #1: Chunking transcripts with dynamic padding...")
    batch_chunk_transcripts(input_folder, output_folder)
    print("Phase 2 #1 completed! All transcripts have been processed.")


# Stage 2: SLM-Assisted Open Coding and Verification

**Author:** Rhesa Muhammad Ramadhan (29024002@mahasiswa.itb.ac.id)  
**License:** GNU AGPL

**Purpose:** Read chunked transcript windows and generate grounded open codes with a locally served small language model.

**Input:** Chunk CSV files from Stage 1, model and prompt settings, and optional prior code definitions for iterative reuse.

**Process:** For each window, build the prompt, query the Ollama model, extract `Code`, `Definition`, and `Text` fields from the response, verify the quoted text against the original window, update the codebook, and log timing and recovery events.

**Output:** Coding result CSV files, codebook CSV files, and run-level logs/statistics saved to `OUTPUT_FOLDER`.

In [ ]:
#CODE: 20250420 - 01:06 PM
import os
import csv
import re
import time
import gc
import torch
from functools import wraps
import errno
from datetime import datetime, timedelta
from tqdm import tqdm
from ollama import chat
from collections import Counter

import subprocess
import psutil
import time
import threading
from functools import wraps

PROGRAM_START_TIME = datetime.now().strftime("%Y%m%d_%H%M%S")

# EDIT DISINI CUY
# Define paths and settings
INPUT_FOLDER = "./stage2_500-word_prompt/qwen2.5-7b/qualitative_w50_ov0.00"
OUTPUT_FOLDER = "./stage2_500-word_prompt/qwen2.5-7b/stage2_w50_ov0.00_win"
BATCH_PROCESSING = True
SINGLE_INPUT_FILE = "chunked_IPWBR_Transcript_Post_100_complete_nopad.csv"

# Debug settings
DEBUG_MODE = True  # Set to False to disable detailed debug outputs

# Model token configurations
MAX_TOKENS_IN = 4096  # Maximum input context length (num_ctx parameter)
MAX_TOKENS_OUT = -1  # Maximum output generation length (num_predict parameter)

# Model configurations with individual temperature settings
MODEL_CONFIGS = {
    "qwen2.5-7b:q4_k_m": {
        "iterations": 1,
        "temperature": 0.0,  
        "max_tokens_in": MAX_TOKENS_IN,  # Model-specific input context length
        "max_tokens_out": MAX_TOKENS_OUT  # Model-specific output length
    },
    # "granite3.1-8b:q4_k_m": {
    #     "iterations": 3,
    #     "temperature": 0.0,
    #     "max_tokens_in": MAX_TOKENS_IN,  # Model-specific input context length
    #     "max_tokens_out": MAX_TOKENS_OUT  # Model-specific output length
    # }
}

# Extract lists for convenience
MODELS = list(MODEL_CONFIGS.keys())
MODEL_ITERATIONS = {model: config["iterations"] for model, config in MODEL_CONFIGS.items()}

class OllamaConnectionError(Exception):
    """Custom exception for Ollama connection issues"""
    pass

def robust_ollama_restart(max_attempts=3, base_delay=5, model_name=None):
    """
    More robust Ollama restart function with progressive delays and health checks
    """
    for attempt in range(max_attempts):
        try:
            # Force kill any existing Ollama processes
            kill_ollama_process()
            
            # Progressive delay between attempts
            delay = base_delay * (2 ** attempt)  # 5s, 10s, 20s...
            print(f"Waiting {delay} seconds before restart attempt {attempt + 1}/{max_attempts}")
            time.sleep(delay)
            
            # Start Ollama service with explicit GPU memory cleanup
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                print("GPU memory cleared pre-startup")
                
            subprocess.Popen(
                ['ollama', 'serve'],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL
            )
            
            # Progressive startup delay
            startup_delay = base_delay * (2 ** attempt)
            print(f"Waiting {startup_delay} seconds for service to start")
            time.sleep(startup_delay)
            
            # Health check with multiple verification steps
            max_health_checks = 3
            for check in range(max_health_checks):
                try:
                    # Try basic command
                    subprocess.run(
                        ['ollama', 'list'], 
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL,
                        timeout=30,
                        check=True
                    )
                    
                    # If model_name provided, try to verify model
                    if model_name:
                        subprocess.run(
                            ['ollama', 'show', model_name],
                            stdout=subprocess.DEVNULL,
                            stderr=subprocess.DEVNULL,
                            timeout=30,
                            check=True
                        )
                    
                    print(f"Health check {check + 1}/{max_health_checks} passed")
                    if check == max_health_checks - 1:
                        print("All health checks passed")
                        return True
                        
                    time.sleep(2)  # Short delay between checks
                    
                except subprocess.SubprocessError as e:
                    print(f"Health check {check + 1} failed: {e}")
                    if check == max_health_checks - 1:
                        break
                    continue
                    
        except Exception as e:
            print(f"Restart attempt {attempt + 1} failed: {str(e)}")
            if attempt == max_attempts - 1:
                print("All restart attempts failed")
                return False
            
            # Additional delay before next attempt
            time.sleep(base_delay)
            
    return False

def handle_connection_error(chunk_address, error, model_name=None):
    """
    Enhanced connection error handler with specific strategies for different error types
    Returns: bool indicating whether to retry
    """
    error_msg = str(error).lower()
    
    if "timeout" in error_msg:
        print(f"\nTimeout occurred at chunk {chunk_address}")
        # For timeouts, first try to kill and restart
        kill_ollama_process()
        time.sleep(5)
        return robust_ollama_restart(model_name=model_name)
        
    elif "actively refused" in error_msg:
        print(f"\nConnection refused at chunk {chunk_address}")
        # First try stopping the model gracefully
        if model_name:
            try:
                subprocess.run(['ollama', 'stop', model_name], 
                             stdout=subprocess.DEVNULL,
                             stderr=subprocess.DEVNULL,
                             timeout=30)
                print(f"Successfully stopped model {model_name}")
                time.sleep(5)  # Give it time to fully stop
                return True
            except Exception as e:
                print(f"Error stopping model gracefully: {e}")
                # If graceful stop fails, fall back to restart
                return robust_ollama_restart(model_name=model_name)
        else:
            return robust_ollama_restart(model_name=model_name)
        
    return False  # Unknown error, don't retry

def timeout(seconds=120):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            result = []
            error = []
            completed = threading.Event()
            
            def worker():
                try:
                    result.append(func(*args, **kwargs))
                except Exception as e:
                    error.append(e)
                finally:
                    completed.set()
            
            thread = threading.Thread(target=worker)
            thread.daemon = True
            thread.start()
            
            # Wait for either completion or timeout
            if not completed.wait(timeout=seconds):
                raise TimeoutError(f"Function call timed out after {seconds} seconds")
            
            if error:
                raise error[0]
            
            return result[0]
        return wrapper
    return decorator

def find_ollama_process():
    """Find the Ollama process if it's running"""
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            if 'ollama' in proc.info['name'].lower():
                return proc
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return None

def kill_ollama_process():
    """Force kill the Ollama process"""
    proc = find_ollama_process()
    if proc:
        try:
            # Kill the process and all its children
            parent = psutil.Process(proc.info['pid'])
            for child in parent.children(recursive=True):
                child.kill()
            parent.kill()
            print("Ollama process killed")
        except (psutil.NoSuchProcess, psutil.AccessDenied) as e:
            print(f"Error killing Ollama process: {e}")

def restart_ollama():
    """Restart the Ollama service"""
    kill_ollama_process()
    time.sleep(5)  # Give it time to fully shut down
    try:
        # Start Ollama service
        subprocess.Popen(['ollama', 'serve'], 
                        stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL)
        time.sleep(10)  # Give it time to start up
        print("Ollama service restarted")
    except Exception as e:
        print(f"Error restarting Ollama: {e}")

@timeout(120)  # 2 minute timeout for the actual inference
def safe_ollama_chat(*args, **kwargs):
    """Execute chat with separate timeout for inference"""
    try:
        return chat(*args, **kwargs)
    except Exception as e:
        raise Exception(f"Ollama chat error: {str(e)}")

def log_event(model_name, iteration, chunk_address, event_type, message, retry_attempt=None, status=None, num_codes=None):
    """
    Log error and success information to file immediately when they occur
    Uses the same timestamp format as run results for consistency
    """
    # Use PROGRAM_START_TIME for the filename to match run results format
    error_log_file = os.path.join(OUTPUT_FOLDER, f"{PROGRAM_START_TIME}_error_logs.txt")
    
    # Create OUTPUT_FOLDER if it doesn't exist
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    try:
        with open(error_log_file, "a", encoding="utf-8") as f:
            f.write(f"\n[{timestamp}] {model_name} - Iteration {iteration}\n")
            f.write(f"Chunk Address: {chunk_address}\n")
            
            if retry_attempt is not None:
                f.write(f"Retry Attempt: {retry_attempt}/{3}\n")  # Hardcoding max_retries=3 for clarity
                f.write(f"Status: {status}\n")
                
                if status == "Success" and num_codes is not None:
                    f.write(f"Codes Generated: {num_codes}\n")
            
            f.write(f"Event Type: {event_type}\n")
            f.write(f"Message: {message}\n")
            f.write("-" * 80 + "\n")
            
            # Force write to disk immediately
            f.flush()
            os.fsync(f.fileno())
            
    except Exception as e:
        print(f"Error writing to log file: {str(e)}")

def debug_print(*args, **kwargs):
    """Wrapper for debug prints that respects DEBUG_MODE setting"""
    if DEBUG_MODE:
        print(*args, **kwargs)

def get_model_shortname(model_name):
    """Extract a shorter version of the model name for file naming"""
    return model_name.split('-')[0]  # e.g., "granite3.1" from "granite3.1-8b:q5_k_m"

def get_datetime_str():
    """Generate a datetime string in the format YYYYMMDD_HHMMSS"""
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def generate_output_filename(input_file, output_folder, model_name, run_number):
    """
    Generate output filename with model name and run number
    Uses a single timestamp for the entire program run
    """
    # Remove 'chunked_' from input filename and get base name
    base_name = os.path.basename(input_file).replace("chunked_", "").replace(".csv", "")
    
    # Create timestamped subfolder based on base_name
    subfolder_name = f"{PROGRAM_START_TIME}_{base_name}"
    subfolder = os.path.join(output_folder, subfolder_name)
    os.makedirs(subfolder, exist_ok=True)
    
    # Generate output filename with model shortname and run number
    model_shortname = get_model_shortname(model_name)
    output_filename = f"{base_name}_{model_shortname}_run_{run_number}.csv"
    
    return os.path.join(subfolder, output_filename)
    
def standardize_code_name(code):
    """
    Standardize code names to prevent duplicates with different capitalization/formatting
    """
    # Convert to lower case and strip whitespace
    standardized = code.lower().strip()
    # Remove any special characters except spaces
    standardized = re.sub(r'[^\w\s]', ' ', standardized)
    # Replace multiple spaces with single space
    standardized = re.sub(r'\s+', ' ', standardized)
    # Capitalize first letter of each word
    standardized = ' '.join(word.capitalize() for word in standardized.split())
    return standardized

def text_exists_in_original(quoted_text, original_text):
    """
    More forgiving text matching that still maintains accuracy
    """
    # Clean up whitespace and convert to lower case
    clean_quote = ' '.join(quoted_text.split()).lower()
    clean_original = ' '.join(original_text.split()).lower()
    
    # Try exact match first
    if clean_quote in clean_original:
        return True
    
    # Remove punctuation and try again
    clean_quote_alphanum = ''.join(c for c in clean_quote if c.isalnum() or c.isspace())
    clean_original_alphanum = ''.join(c for c in clean_original if c.isalnum() or c.isspace())
    
    if clean_quote_alphanum in clean_original_alphanum:
        return True
    
    # Try with normalized whitespace
    clean_quote_norm = ' '.join(clean_quote_alphanum.split())
    clean_original_norm = ' '.join(clean_original_alphanum.split())
    
    return clean_quote_norm in clean_original_norm

def extract_codes_from_response(response_text, original_text, existing_codes=None):
    """
    Extract code_name, code_definition, and supporting quote from the LLM's output.
    Stores definitions and updates if the code already exists.
    """
    if existing_codes is None:
        existing_codes = {}

    codes = []
    blocks = re.split(r'\n\s*(?=Code:)', response_text)

    for block in blocks:
        if not block.strip() or block.lower().startswith('example') or block.lower().startswith('guid'):
            continue

        try:
            code_match = re.search(r'Code:\s*(.+?)(?:\n|$)', block)
            def_match = re.search(r'Definition:\s*(.+?)(?:\n|$)', block)
            text_match = re.search(r'Text:\s*"(.+?)"', block)
            if not text_match:
                text_match = re.search(r'Text:\s*(.+?)(?:\n|$)', block)

            if not code_match or not def_match or not text_match:
                if DEBUG_MODE:
                    print("\n[DEBUG] Skipped block due to missing element:")
                    print("────────────────────────────────────────────")
                    print(block.strip())
                    print("────────────────────────────────────────────\n")
                continue

            code_name = standardize_code_name(code_match.group(1).strip())
            code_definition = def_match.group(1).strip()
            quoted_text = text_match.group(1).strip().strip('"')

            if text_exists_in_original(quoted_text, original_text):
                codes.append({
                    "code_name": code_name,
                    "code_definition": code_definition,
                    "data": quoted_text
                })
                existing_codes[code_name] = code_definition
            else:
                debug_print(f"DEBUG - Text match failed: {quoted_text}")

        except Exception as e:
            debug_print(f"Error processing block: {str(e)}")
            continue

    return codes, existing_codes

def build_prompt(chunk_text, existing_codes=None):
    """
    Construct a single system prompt that includes both instructions and data.
    The user prompt is empty.
    """

    if existing_codes:
        existing_codes_str = "\n".join(f"- {k}: {v}" for k, v in sorted(existing_codes.items()))
        reuse_block = f""" {existing_codes_str} """
    else:
        reuse_block = ""

    instruction_block = f"""
**Role:** AI Qualitative Assistant performing iterative open coding on transcript windows.

**Objective:** Identify and code patterns on the impact of programming workshops (R, Python, Git, Unix) on biomedical researchers' computational reproducibility workflows.

**Simulated Perspective:** Focus on practical workflow changes related to reproducibility (scripting, version control, sharing, open source). Assume a research support context valuing practical outcomes. Be sensitive to themes of Tool Adoption/Use, Workflow Changes, Reproducibility Practices, Enablers/Barriers, Learning/Literacy, Future Intentions – use these *only* to interpret relevance of emergent codes.

**Guiding Principles:** **Data First!** Codes must be grounded in text. Be inductive – let patterns emerge. Aim for conceptual labels, not just description.

**Input for this Window:**

1.  **Current Transcript Window:** "{chunk_text}"
2.  **List of Codes from Previous Window(s):** {reuse_block.strip()}

**Process:**

1.  **Read & Excerpt:** Read the window, identify meaningful excerpts relevant to the objective. Ignore filler.
2.  **Code Each Excerpt:** For each excerpt:
    *   **A. Initial Code:** What's the core action/idea? Generate 1-2 grounded codes (in vivo or descriptive phrase).
    *   **B. Check Prior:** Does the excerpt's core meaning strongly match a **Definition** from the prior code list?
    *   **C. Assign/Generate:**
        *   If YES match: Reuse the prior **Code Name**. Check for nuance (D).
        *   If NO match: Use the best *new* grounded code from A. Provide a concise, one-sentence **Definition** grounded in the excerpt.
    *   **D. Nuance Check (Optional):** If reusing a prior code, can the **Definition** be slightly revised (one sentence still) to add nuance from this excerpt without changing core meaning? If yes, provide **Revised Definition**.
    *   **Multiple Codes:** If excerpt has multiple distinct relevant ideas, repeat A-C/D for each. Each requires a separate output entry.

**Output Format (Mandatory - One entry per code assigned):**

```
Code: [Code Name (Reused or New)]
Definition: [One-sentence definition (Existing, New, or Revised)]
Text: "[Exact quote excerpt]"
```

**Constraints:** Ground all codes/definitions in text. Prioritize induction. Use perspective lens carefully for relevance/interpretation only. No external info. One code per output entry.

**Examples:**

*   **New Code Example:**
    *   Prior Codes: `[Code: Time Barrier | Definition: Lack of time hinders practice.]`
    *   Window Text: "Seeing the R script made me realize clicking menus isn't reproducible."
    *   Output:
        ```
        Code: Realizing Scripting Value
        Definition: Participant understands scripting enhances reproducibility over GUI actions.
        Text: "[Seeing the R script made me realize clicking menus isn't reproducible.]"
        ```
*   **Reuse Code Example:**
    *   Prior Codes: `[Code: Time Barrier | Definition: Lack of time hinders learning/implementing.]`, `[Code: Realizing Scripting Value | Definition: Participant understands scripting enhances...]`
    *   Window Text: "The main hurdle to using Git is finding time to practice."
    *   Output:
        ```
        Code: Time Barrier
        Definition: Lack of time hinders learning or implementing new practices.
        Text: "[The main hurdle to using Git is finding time to practice.]"
        ```
*   **Reuse & Revise Definition Example:**
    *   Prior Codes: `[Code: Collaboration Barrier | Definition: Difficulties working with collaborators.]`, `[Code: Time Barrier | Definition: Lack of time hinders... ]`
    *   Window Text: "PI agrees GitHub is good, but insists on email because it's 'simpler for them'."
    *   Output:
        ```
        Code: Collaboration Barrier
        Definition: Difficulties implementing new tools due to collaborator/supervisor resistance or preference.
        Text: "[PI agrees GitHub is good, but insists on email because it's 'simpler for them'.]"
        ```
"""

    return instruction_block.strip(), ""  # user prompt is empty

    
def process_chunk(chunk_text, window_address, code_id_start, model_name, existing_codes=None, iteration="N/A"):
    model_config = MODEL_CONFIGS[model_name]
    max_tokens_in = model_config["max_tokens_in"]
    max_tokens_out = model_config["max_tokens_out"]
    temperature = model_config["temperature"]

    # Modular prompt construction
    system_prompt, prompt = build_prompt(chunk_text, existing_codes)
    
    max_retries = 3
    current_try = 0
    progressive_timeout = 120  # Start with 2 minutes

    while current_try < max_retries:
        try:
            current_try += 1  # Increment at start of attempt
            current_timeout = progressive_timeout * (1.5 ** (current_try - 1))
            
            if current_try > 1:  # Log retry attempt start
                print(f"\nRetry attempt {current_try} for chunk {window_address}")
            
            print(f"Attempting chat call for chunk {window_address} (try {current_try}/{max_retries}) with {int(current_timeout)}s timeout")
            
            try:
                # Run the chat call with the progressive timeout
                @timeout(seconds=int(current_timeout))
                def run_chat():
                    return safe_ollama_chat(
                        model=model_name,
                        messages=[{
                            "role": "system",
                            "content": system_prompt
                        }, {
                            "role": "user",
                            "content": prompt
                        }],
                        options={
                            "temperature": temperature,
                            "num_predict": max_tokens_out,
                            "num_ctx": max_tokens_in
                        }
                    )
                
                response = run_chat()
                
                # Process successful response
                print(f"Completed chat call for chunk {window_address}")
                response_text = response["message"]["content"].strip()
                
                debug_print("\nDEBUG - LLM Response:")
                debug_print(response_text)
                debug_print("\nDEBUG - Attempting to extract codes...")
                
                codes, updated_existing_codes = extract_codes_from_response(
                    response_text, 
                    chunk_text, 
                    existing_codes
                )
                
                if not codes:
                    # Log empty chunk result immediately
                    log_event(
                        model_name,
                        iteration,
                        window_address,
                        "Empty Result",
                        "No valid codes extracted from response",
                        retry_attempt=current_try,
                        status="Empty Chunk"
                    )
                    debug_print("\nDEBUG - No valid codes extracted from response")
                else:
                    # Log successful code generation immediately
                    # log_event(
                    #     model_name,
                    #     iteration,
                    #     window_address,
                    #     "Success",
                    #     "Successfully generated codes",
                    #     retry_attempt=current_try,
                    #     status="Success",
                    #     num_codes=len(codes)
                    # )
                    debug_print(f"\nDEBUG - Extracted {len(codes)} codes")
                
                # Add window_address and code_id to each code
                results = []
                for i, code in enumerate(codes, start=code_id_start):
                    code['window_address'] = str(window_address)
                    code['code_id'] = f"cd{i:05d}"
                    results.append(code)
                    
                return results, updated_existing_codes
                
            except TimeoutError:
                log_event(
                    model_name,
                    iteration,
                    window_address,
                    "Timeout",
                    f"Operation timed out after {int(current_timeout)}s",
                    retry_attempt=current_try,
                    status="Retry" if current_try < max_retries else "Failed"
                )
                print(f"\nTimeout occurred for chunk {window_address}")
                
                # Kill Ollama and clear memory before restart
                kill_ollama_process()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    print("GPU memory cleared")
                
                # Wait longer before restart on each retry
                wait_time = 5 * current_try
                print(f"Waiting {wait_time} seconds before restart...")
                time.sleep(wait_time)
                
                # Try to restart Ollama
                if robust_ollama_restart(model_name=model_name):
                    continue
                else:
                    print(f"\nERROR: Failed to recover from timeout for chunk {window_address}")
                    return [], existing_codes
                
            except Exception as e:
                error_msg = str(e)
                log_event(
                    model_name,
                    iteration,
                    window_address,
                    "Chat Error",
                    error_msg,
                    retry_attempt=current_try,
                    status="Retry" if current_try < max_retries else "Failed"
                )
                print(f"Error in chat operation: {error_msg}")
                
                # Try to handle the error with our enhanced error handler
                if handle_connection_error(window_address, e, model_name):
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        print("GPU memory cleared")
                    continue
                else:
                    if current_try >= max_retries:
                        print(f"Error processing chunk with window address {window_address}: {error_msg}")
                        return [], existing_codes or set()
                    continue
                
            finally:
                # Always perform cleanup
                try:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                        debug_print("GPU memory cleared")
                except Exception as e:
                    debug_print(f"Memory cleanup warning: {str(e)}")
                
        except Exception as e:
            # Log unexpected errors immediately
            log_event(
                model_name,
                iteration,
                window_address,
                "Unexpected Error",
                str(e),
                retry_attempt=current_try,
                status="Failed"
            )
            print(f"Unexpected error in process_chunk: {str(e)}")
            return [], existing_codes or set()

    # If we somehow get here (shouldn't happen due to returns in the loops)
    return [], existing_codes

def process_file(input_file, current_code_id=1, run_stats=None):
    """
    Process a single file with multiple models and iterations
    Returns the next available code_id and updated run_stats
    """
    
    try:
        # Initialize run_stats if None
        if run_stats is None:
            run_stats = {}
        
        # Ensure each model has all required fields
        for model in MODELS:
            if model not in run_stats:
                run_stats[model] = {
                    "iterations": {},
                    "total_codes": 0,
                    "unique_codes": 0,
                    "cumulative_codes": set()
                }
            elif "cumulative_codes" not in run_stats[model]:
                run_stats[model]["cumulative_codes"] = set()

            # Initialize file_stats if not present
            if "file_stats" not in run_stats[model]:
                run_stats[model]["file_stats"] = {}

            # Initialize this specific file's stats
            if input_file not in run_stats[model]["file_stats"]:
                run_stats[model]["file_stats"][input_file] = {}

        full_input_path = os.path.join(INPUT_FOLDER, input_file)
        if not os.path.exists(full_input_path):
            print(f"Input file {full_input_path} does not exist.")
            return current_code_id, run_stats

        print(f"\nProcessing chunked transcript '{input_file}'...")
        
        # Process for each model and iteration
        for model_name in MODELS:
            try:
                iterations = MODEL_CONFIGS[model_name]["iterations"]
                print(f"\nStarting processing with model: {model_name}")
                print(f"Temperature: {MODEL_CONFIGS[model_name]['temperature']}")
                print(f"Planning {iterations} iterations")
                
                for iteration in range(1, iterations + 1):

                    # Add this line ↓
                    existing_codes = {}
                    
                    try:
                        print(f"\nProcessing iteration {iteration}/{iterations} with {model_name}")
                        print(f"Starting from code_id: cd{current_code_id:05d}")

                        output_data = []
                        chunk_times = []
                        run_start_time = time.time()

                        with open(full_input_path, mode="r", encoding="utf-8") as infile:
                            reader = csv.DictReader(infile, delimiter=";")
                            chunks = list(reader)

                        for row in tqdm(chunks, desc=f"Processing {model_name} - Run {iteration}", unit="chunk"):
                            chunk_start_time = time.time()
                            window_address = row["window_address"]
                            chunked_text = row["chunked_text"]
                            
                            results, updated_codes = process_chunk(
                                chunk_text=chunked_text,
                                window_address=window_address,
                                code_id_start=current_code_id,
                                model_name=model_name,
                                existing_codes=existing_codes,
                                iteration=iteration
                            )
                            
                            chunk_end_time = time.time()
                            chunk_duration = chunk_end_time - chunk_start_time
                            chunk_times.append(chunk_duration)
                            
                            if results:
                                output_data.extend(results)
                                current_code_id += len(results)

                        # Save output if we have data
                        if output_data:
                            output_file = generate_output_filename(
                                input_file=input_file,
                                output_folder=OUTPUT_FOLDER,
                                model_name=model_name,
                                run_number=iteration
                            )
                            
                            with open(output_file, mode="w", encoding="utf-8", newline="") as outfile:
                                fieldnames = ["code_id", "window_address", "code_name", "data"]
                                writer = csv.DictWriter(outfile, fieldnames=fieldnames, delimiter=";")
                                writer.writeheader()
                                
                                cleaned_data = []
                                for row in output_data:
                                    cleaned_row = {
                                        "code_id": row["code_id"],
                                        "window_address": row["window_address"],
                                        "code_name": row["code_name"],
                                        "data": row["data"]
                                    }
                                    cleaned_data.append(cleaned_row)
                                
                                writer.writerows(cleaned_data)

                            # Calculate current run's unique codes
                            current_run_codes = set(code["code_name"] for code in output_data)
                            unique_codes_this_run = len(current_run_codes)
                            
                            # Get the set of codes before this iteration
                            previous_codes = run_stats[model_name]["cumulative_codes"].copy()
                            
                            # Find new unique codes by comparing with all previous codes
                            new_codes = current_run_codes - previous_codes
                            new_unique_codes = len(new_codes)
                            
                            # Update the cumulative set with new codes
                            run_stats[model_name]["cumulative_codes"].update(current_run_codes)
                            
                            # Debug information
                            if DEBUG_MODE:
                                print("\nDEBUG - Code Sets:")
                                print(f"Current run unique codes ({len(current_run_codes)}): {sorted(current_run_codes)}")
                                print(f"Cumulative unique codes ({len(run_stats[model_name]['cumulative_codes'])}): {sorted(run_stats[model_name]['cumulative_codes'])}")
                                
                                if iteration > 1:
                                    print("\nDEBUG - New Codes Analysis:")
                                    if new_codes:
                                        print("New codes found in this run:")
                                        for code in sorted(new_codes):
                                            print(f"  + {code}")
                                    else:
                                        print("No new codes found in this run")
                                    
                                    unused_codes = previous_codes - current_run_codes
                                    print("\nPrevious codes not used in this run:")
                                    for code in sorted(unused_codes):
                                        print(f"  - {code}")
                                    
                                    print("\nDEBUG - Code Movement Analysis:")
                                    print(f"Previous unique codes: {len(previous_codes)}")
                                    print(f"New codes this run: {len(new_codes)}")
                                    print(f"Total unique codes: {len(run_stats[model_name]['cumulative_codes'])}")
                            
                            print(f"\nCompleted! Output saved to {output_file}")

                            # === Save Codebook CSV ===
                            codebook_file = output_file.replace(".csv", "_codebook.csv")
                            try:
                                with open(codebook_file, mode="w", encoding="utf-8", newline="") as cb:
                                    writer = csv.DictWriter(cb, fieldnames=["code_name", "code_definition"], delimiter=";")
                                    writer.writeheader()
                            
                                    unique_code_defs = {}
                                    for code in output_data:
                                        c = code["code_name"]
                                        d = code["code_definition"]
                                        unique_code_defs[c] = d  # overwrite with latest version
                            
                                    for code_name, code_definition in sorted(unique_code_defs.items()):
                                        writer.writerow({"code_name": code_name, "code_definition": code_definition})
                            
                                print(f"Codebook saved to {codebook_file}")
                            except Exception as e:
                                print(f"Failed to save codebook: {e}")
                            
                            print(f"Codes generated in this run: {len(output_data)}")
                            print(f"Unique codes in this run: {unique_codes_this_run}")
                            print(f"New unique codes (compared to previous runs): {new_unique_codes}")
                            print(f"Total unique codes across all runs: {len(run_stats[model_name]['cumulative_codes'])}")
                            
                            if iteration > 1:
                                reused_codes = len(current_run_codes.intersection(previous_codes))
                                print(f"Reused codes in this run: {reused_codes}")
                                unused_previous = len(previous_codes - current_run_codes)
                                print(f"Previous codes not used in this run: {unused_previous}")
                        
                        # Safely accumulate run duration across files for the same iteration
                        prev_duration = run_stats[model_name]["iterations"].get(iteration, {}).get("run_duration", 0)
                        
                        run_stats[model_name]["iterations"][iteration] = {
                            "total_codes": run_stats[model_name]["iterations"].get(iteration, {}).get("total_codes", 0) + len(output_data),
                            "unique_codes": run_stats[model_name]["iterations"].get(iteration, {}).get("unique_codes", 0) + unique_codes_this_run,  # this won't double-count names, but for consistency
                            "new_unique_codes": run_stats[model_name]["iterations"].get(iteration, {}).get("new_unique_codes", 0) + new_unique_codes,
                            "run_duration": prev_duration + (time.time() - run_start_time),
                            "chunks_processed": run_stats[model_name]["iterations"].get(iteration, {}).get("chunks_processed", 0) + len(chunks),
                            "avg_time_per_chunk": 0,  # Will be recalculated in dump_run_results
                            "total_chunk_times": run_stats[model_name]["iterations"].get(iteration, {}).get("total_chunk_times", []) + chunk_times,
                            "code_distribution": dict(Counter(code["code_name"] for code in output_data))  # Will overwrite — okay for now
                        }

                        # NEW: Add per-transcript tracking
                        if "file_stats" not in run_stats[model_name]:
                            run_stats[model_name]["file_stats"] = {}
                        
                        if input_file not in run_stats[model_name]["file_stats"]:
                            run_stats[model_name]["file_stats"][input_file] = {}
                        
                        run_stats[model_name]["file_stats"][input_file][iteration] = {
                            "total_codes": len(output_data),
                            "unique_codes": unique_codes_this_run,
                            "new_unique_codes": new_unique_codes,
                            "run_duration": time.time() - run_start_time,
                            "chunks_processed": len(chunks),
                            "avg_time_per_chunk": sum(chunk_times) / len(chunk_times) if chunk_times else 0,
                            "total_chunk_times": chunk_times,
                            "code_distribution": dict(Counter(code["code_name"] for code in output_data))
                        }
                        
                        # Update total statistics
                        run_stats[model_name]["total_codes"] = sum(
                            iter_stats["total_codes"] 
                            for iter_stats in run_stats[model_name]["iterations"].values()
                        )
                        run_stats[model_name]["unique_codes"] = len(run_stats[model_name]["cumulative_codes"])
                        
                    except Exception as e:
                        print(f"\nError in iteration {iteration} for {model_name}: {str(e)}")
                        # Try to restart Ollama service
                        if robust_ollama_restart():
                            print(f"Successfully restarted Ollama after iteration error")
                        continue
                        
            except Exception as e:
                print(f"\nError in model {model_name}: {str(e)}")
                # Try to restart Ollama service
                if robust_ollama_restart():
                    print(f"Successfully restarted Ollama after model error")
                continue

    except KeyboardInterrupt:
        print("\nReceived interrupt signal. Cleaning up...")
        kill_ollama_process()
        print("Ollama process killed. You can safely restart the notebook.")
        raise
        
    except Exception as e:
        print(f"\nCritical error in process_file: {str(e)}")
        kill_ollama_process()
        raise
        
    finally:
        # Ensure cleanup happens
        try:
            kill_ollama_process()
        except Exception as e:
            print(f"Warning: Cleanup failed: {str(e)}")
    
    return current_code_id, run_stats

def dump_run_results(run_stats, config_info):
    """
    Dump run statistics and configuration into a timestamped log file in the output folder
    Uses PROGRAM_START_TIME for consistent timestamp across the program
    """
    from datetime import datetime
    
    # Use PROGRAM_START_TIME for the filename
    log_filename = os.path.join(OUTPUT_FOLDER, f"{PROGRAM_START_TIME}_run_results.txt")
    
    # Ensure OUTPUT_FOLDER exists first
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    
    try:
        with open(log_filename, "w", encoding="utf-8") as f:
            # Write timestamp and program info
            f.write(f"Run Results - Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("="*80 + "\n\n")
            
            # Write configuration
            f.write("CONFIGURATION\n")
            f.write("-"*50 + "\n")
            for key, value in config_info.items():
                f.write(f"{key}: {value}\n")
            
            # Add carryover mode specific details
            if config_info.get("Code Carryover Mode") == "SAME_LLM":
                f.write(f"\nCarryover Threshold Details:\n")
                f.write(f"- Independent coding until iteration {config_info['Carryover Threshold']}\n")
                f.write(f"- Code carryover active after iteration {config_info['Carryover Threshold']}\n")
            f.write("\n")
            
            # Write run statistics
            f.write("RUN STATISTICS\n")
            f.write("-"*50 + "\n")
            for model, stats in run_stats.items():
                f.write(f"\nModel: {model}\n")
                f.write(f"Total Cumulative Unique Codes: {len(stats.get('cumulative_codes', set()))}\n")
                
                if not stats["iterations"]:  # Skip if no iterations were recorded
                    f.write("  No iterations recorded for this model\n")
                    continue
                    
                f.write("  Iterations:\n")
                for iter_num, iter_stats in sorted(stats["iterations"].items()):
                    f.write(f"    Run {iter_num}:\n")
                    # Add carryover mode indicator
                    if config_info.get("Code Carryover Mode") == "SAME_LLM":
                        mode = "Independent" if iter_num <= config_info['Carryover Threshold'] else "Carryover"
                        f.write(f"      Mode: {mode}\n")
                    
                    # Basic statistics
                    f.write(f"      Total codes generated: {iter_stats['total_codes']}\n")
                    f.write(f"      Unique codes in this run: {iter_stats['unique_codes']}\n")
                    f.write(f"      New unique codes: {iter_stats['new_unique_codes']}\n")
                    
                    # Timing information
                    duration_str = str(timedelta(seconds=int(iter_stats['run_duration'])))
                    total_chunks = iter_stats['chunks_processed']
                    total_time = iter_stats['run_duration']
                    avg_chunk_time = total_time / total_chunks if total_chunks else 0
                    
                    f.write(f"      Run duration: {duration_str}\n")
                    f.write(f"      Chunks processed: {iter_stats['chunks_processed']}\n")
                    f.write(f"      Average time per chunk: {avg_chunk_time:.2f} seconds\n")
                    f.write(f"      Processing speed: {1/avg_chunk_time:.2f} chunks/second\n" if avg_chunk_time else "      Processing speed: N/A\n")
                    
                    # Performance percentiles
                    if iter_stats.get('total_chunk_times'):
                        times = sorted(iter_stats['total_chunk_times'])
                        p50 = times[len(times)//2]
                        p90 = times[int(len(times)*0.9)]
                        p95 = times[int(len(times)*0.95)]
                        f.write(f"      Processing time percentiles:\n")
                        f.write(f"        50th percentile: {p50:.2f} seconds\n")
                        f.write(f"        90th percentile: {p90:.2f} seconds\n")
                        f.write(f"        95th percentile: {p95:.2f} seconds\n")
                    
                    # Code distribution
                    if iter_stats.get('code_distribution'):
                        f.write("      Code distribution:\n")
                        total_codes = sum(iter_stats['code_distribution'].values())
                        sorted_codes = sorted(
                            iter_stats['code_distribution'].items(),
                            key=lambda x: (-x[1], x[0])  # Sort by count (desc) then name (asc)
                        )
                        for code, count in sorted_codes:
                            percentage = (count / total_codes) * 100
                            f.write(f"        {code}: {count} ({percentage:.1f}%)\n")
                
                # Per-transcript statistics
                f.write("\n  Transcript Files:\n")
                for filename, file_stats in sorted(stats.get("file_stats", {}).items()):
                    f.write(f"    File: {filename}\n")
                    for run_num, run_data in sorted(file_stats.items()):
                        f.write(f"      Run {run_num}:\n")
                        f.write(f"        Total codes: {run_data.get('total_codes', 0)}\n")
                        f.write(f"        Unique codes: {run_data.get('unique_codes', 0)}\n")
                        f.write(f"        New unique codes: {run_data.get('new_unique_codes', 0)}\n")
                        dur_str = str(timedelta(seconds=int(run_data.get('run_duration', 0))))
                        f.write(f"        Run duration: {dur_str}\n")
                        f.write(f"        Chunks processed: {run_data.get('chunks_processed', 0)}\n")
                        avg_time = run_data.get('avg_time_per_chunk', 0)
                        f.write(f"        Average time per chunk: {avg_time:.2f} seconds\n")
                        f.write(f"        Processing speed: {1/avg_time:.2f} chunks/second\n")

                        if run_data.get("total_chunk_times"):
                            times = sorted(run_data["total_chunk_times"])
                            p50 = times[len(times)//2]
                            p90 = times[int(len(times)*0.9)]
                            p95 = times[int(len(times)*0.95)]
                            f.write(f"        Processing time percentiles:\n")
                            f.write(f"          50th percentile: {p50:.2f} seconds\n")
                            f.write(f"          90th percentile: {p90:.2f} seconds\n")
                            f.write(f"          95th percentile: {p95:.2f} seconds\n")

                        if run_data.get("code_distribution"):
                            f.write("        Code distribution:\n")
                            total_codes = sum(run_data["code_distribution"].values())
                            sorted_codes = sorted(
                                run_data["code_distribution"].items(),
                                key=lambda x: (-x[1], x[0])
                            )
                            for code, count in sorted_codes:
                                percentage = (count / total_codes) * 100
                                f.write(f"          {code}: {count} ({percentage:.1f}%)\n")

                # Summary statistics for the model
                f.write(f"\n  Model Summary:\n")
                f.write(f"    Total codes generated across all iterations: {stats['total_codes']}\n")
                f.write(f"    Total unique codes: {stats['unique_codes']}\n")
                
                # Add cumulative codes list if in debug mode
                if DEBUG_MODE:
                    f.write("\n  DEBUG - Cumulative Unique Codes:\n")
                    for code in sorted(stats.get('cumulative_codes', set())):
                        f.write(f"    - {code}\n")
        
        print(f"\nRun results have been saved to: {log_filename}")
        return True
    except Exception as e:
        print(f"\nError writing run results to {log_filename}: {str(e)}")
        return False

def main():
    # Initialize statistics collection
    run_stats = {model: {
        "iterations": {},
        "total_codes": 0,
        "unique_codes": 0
    } for model in MODELS}
    
    # Collect configuration info
    config_info = {
        "Debug Mode": DEBUG_MODE,
        "Batch Processing": BATCH_PROCESSING,
        "Input Folder": INPUT_FOLDER,
        "Output Folder": OUTPUT_FOLDER,
        "Single Input File": SINGLE_INPUT_FILE if not BATCH_PROCESSING else "N/A",
        "Max Input Tokens": MAX_TOKENS_IN,
        "Max Output Tokens": MAX_TOKENS_OUT
    }
    
    # Add model configurations
    for model, config in MODEL_CONFIGS.items():
        config_info[f"Model {model} Config"] = (
            f"Iterations: {config['iterations']}, "
            f"Temperature: {config['temperature']}, "
            f"Max Input Tokens: {config['max_tokens_in']}, "
            f"Max Output Tokens: {config['max_tokens_out']}"
        )
    
    # Create OUTPUT_FOLDER at the start
    try:
        os.makedirs(OUTPUT_FOLDER, exist_ok=True)
        print(f"\nCreated/verified output folder: {OUTPUT_FOLDER}")
    except Exception as e:
        print(f"\nError creating output folder {OUTPUT_FOLDER}: {str(e)}")
        return
    
    # Print configuration
    print("\nCurrent Configuration:")
    print(f"Debug Mode: {DEBUG_MODE}")
    
    if BATCH_PROCESSING:
        print("\nBatch processing enabled.")
        input_files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".csv")]
    else:
        print(f"\nProcessing single file: {SINGLE_INPUT_FILE}")
        input_files = [SINGLE_INPUT_FILE]
    
    current_code_id = 1  # Initialize the global code_id counter
    
    # Process each file and collect statistics
    for input_file in input_files:
        print(f"\nStarting processing of {input_file}")
        current_code_id, run_stats = process_file(input_file, current_code_id, run_stats)
        print(f"Completed processing {input_file}")
        print(f"Next available code_id: cd{current_code_id:05d}")
    
    # Dump results after all processing is complete
    if not dump_run_results(run_stats, config_info):
        print("\nFailed to dump run results. Please check the output folder permissions and path.")
    
if __name__ == "__main__":
    main()